In [1]:
from pathlib import Path
from xlsxwriter import Workbook
import numpy as np
import pandas as pd
import pyodbc
import sys
import os
import time
from datetime import date
import re
import warnings as wr
wr.filterwarnings('ignore')

In [2]:
seasons_as10 = {1:'Kharif', 2:'Phase-I', 3:'Phase-II', 4:'Phase-III', 5:'Summer'}
seasons_as20 = {1:'Kharif', 2:'E.Kharif', 3:'L.Kharif', 4:'Rabi', 5:'Summer'}

In [3]:
def clean_rc123(year_int, season_cd, planning, b4_cod):
    #set database file path
    folder_path = Path('./software')
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            if season_nm.lower() in file_path.name.lower():
                accdb_path = "./" + "/".join(str(file_path).split("\\"))
    
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={accdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - RC1
    season_nm = seasons[season_cd]
    st_cd = 1
    state_cd = 33
    year = 2024
    planning = 260
    b4_cod = 250
    season_cd = 2
    ddg = 2510
    ddtrs = 2810
    
    cols1 = ['RC', 'ST', 'YR', 'SESON', 'NOVP', 'NOVCOD', 'STAT', 'DIST', 'STRA',
           'VILL', 'EPC', 'NOVPC', 'NOVTRS', 'CADC', 'TLC', 'MAC', 'MUC', 'DDG',
           'SGC', 'GCC', 'LFOG', 'ROG', 'DDTRS', 'TRSSC', 'TRSSF', 'DCKC', 'HSSN',
           'GEOA', 'TCROPA', 'TNSSN', 'TGEOASN', 'RC61', 'RC62', 'RC63', 'REJ']
    
    sql_query1 = 'SELECT * FROM "Copy Of RC1"'
    df1 = pd.read_sql(sql_query1, conn)[cols1]
    
    df1 = (df1
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd")
        .assign(
            REJ = lambda x : np.where(x['REJ']==1, 1, 0)
            ,NOVPC1 = lambda x : x['NOVPC'].fillna(1)
            ,NOVPC = lambda x : np.where(x['NOVPC1']==0, 1, x['NOVPC1'])
            ,NOVTRS1 = lambda x : x['NOVTRS'].fillna(1)
            ,NOVTRS = lambda x : np.where(x['NOVTRS1']==0, 1, x['NOVTRS1'])
            ,CADC = lambda x : np.where(np.isin(x['CADC'], [1,2,3,4]), x['CADC'], 9)
            ,TLC = lambda x : np.where(x['CADC']==3, 0, np.where(np.isin(x['CADC'], [4,9]), 8, x['TLC']))
            ,MAC = lambda x : np.where(x['MAC']==1, 1, 0)
            ,MUC = lambda x : np.where(x['MAC']==1, x['MUC'], None)
            ,TRSSC = lambda x : np.where(x['TRSSC']==3, 8, x['TRSSC'])
            ,NOVP = planning
            ,DDG = ddg
            ,DDTRS = ddtrs
            ,TCROPA = lambda x : x['TCROPA'].fillna(0)
            ,GEOA = lambda x : x['GEOA'].fillna(0)
            ,YR = year+1
            ,NOVCOD = b4_cod
            ,LFOG = lambda x : np.where(~np.isin(x['LFOG'], [0,1]), None, x['LFOG'])
            ,ROG = lambda x : np.where(~np.isin(x['ROG'], [0,1]), None, x['ROG'])
            ,TRSSF = lambda x : np.where(x['TRSSF'].isna(), None, x['TRSSF'])
        )
        .drop(columns = ['NOVPC1','NOVTRS1'])
    )
    
    non_rej_rc1 = df1[df1['REJ']!=1]
    rej_rc1 = df1[df1['REJ']==1].assign(
    	EPC = None
    	,NOVPC = None
    	,NOVTRS = None
    	,CADC = None
    	,TLC = None
    	,MAC = None
    	,MUC = None
    	,DDG = None
    	,SGC = None
    	,GCC = None
    	,LFOG = None
    	,ROG = None
    	,DDTRS = None
    	,TRSSC = None
    	,TRSSF = None
    	,DCKC = None
    	,HSSN = None
    	,GEOA = None
    	,TCROPA = None
    	,TNSSN = None
    	,TGEOASN = None
    	,RC61 = None
    	,RC62 = None
    	,RC63 = None
    )
    rc1 = pd.concat([non_rej_rc1, rej_rc1], ignore_index=True)
    
    if rc1['HSSN'].isna().sum() > 0:
        print('='*50)
        print('In RC1, HSSN is Blank. Please Correct...')
        print('='*50)
    if rc1[rc1['HSSN'] == 0].shape[0] > 0:
        print('='*50)
        print('In RC1, HSSN is Zero. Please Correct...')
        print('='*50)
    if rc1['GEOA'].isna().sum() > 0:
        print('='*50)
        print('In RC1, GEOA is Blank. Please Correct...')
        print('='*50)
    if rc1[rc1['GEOA'] == 0].shape[0] > 0:
        print('='*50)
        print('In RC1, GEOA is Zero. Please Correct...')
        print('='*50)
    
    
    rc1.to_excel(excel_writer='RC1237_clean.xlsx', sheet_name='RC1', index=False, engine='openpyxl')
    
    
    
    
    
    
    
    #get data from the table - RC2
    cols2 = ['RC', 'ST', 'YR', 'SESON', 'STAT', 'DIST', 'STRA', 'VILL', 'EPC', 
             'HSSN', 'TNSSN', 'CROP', 'VARC', 'ARSU', 'ARSI', 'ARPU', 'ARPI']
    
    sql_query2 = 'SELECT * FROM "Copy Of RC2"'
    df2 = pd.read_sql(sql_query2, conn)[cols2]
    df2 = (df2
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd")
        .assign(
            YR = year+1
            ,VARC = lambda x : np.where(x['CROP']==0, 0, x['VARC'])
            ,CROP = lambda x : np.where(x['VARC']==0, 0, x['CROP'])
        )
    )
    rc2 = df2.copy()
    
    with pd.ExcelWriter('RC1237_clean.xlsx', mode='a', engine='openpyxl') as w2:
        rc2.to_excel(w2, sheet_name='RC2', index=False)
    
    
    
    
    
    
    
    #get data from the table - RC3
    cols3 = ['RC', 'ST', 'YR', 'SESON', 'STAT', 'DIST', 'STRA',
             'VILL', 'SN', 'CROP', 'VARC', 'IRRC', 'ERC']
    
    sql_query3 = 'SELECT * FROM "Copy Of RC3"'
    df3 = pd.read_sql(sql_query3, conn)[cols3]
    df3 = (df3
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd")
        .assign(
            YR = year+1
            ,VARC = lambda x : np.where(x['CROP']==0, 0, x['VARC'])
            ,CROP = lambda x : np.where(x['VARC']==0, 0, x['CROP'])
        )
    )
    
    #creating schedule unique id in RC1
    rc1['unique_id'] = rc1['STAT'].astype(str) + "_" + rc1['SESON'].astype(str) + "_" + rc1['ST'].astype(str) + "_" + rc1['STRA'].astype(str) + "_" + rc1['VILL'].astype(str)+ "_" + rc1['DIST'].astype(str)
    
    
    #creating schedule unique id in df3
    df3['unique_id'] = df3['STAT'].astype(str) + "_" + df3['SESON'].astype(str) + "_" + df3['ST'].astype(str) + "_" + df3['STRA'].astype(str) + "_" + df3['VILL'].astype(str)+ "_" + df3['DIST'].astype(str)
    
    #Left Join on df3 with rc1_epc1
    rc1_epc = rc1[['unique_id','EPC']]
    rc3 = pd.merge(df3, rc1_epc, how='left', left_on='unique_id', right_on='unique_id')
    rc3 = rc3[rc3['EPC'] == 1].drop(columns=['unique_id','EPC'])
    
    with pd.ExcelWriter('RC1237_clean.xlsx', mode='a', engine='openpyxl') as w3:
        rc3.to_excel(w3, sheet_name='RC3', index=False)
    
    
    
    
    
    conn.close()
    return rc1, rc2, rc3

In [6]:
year = 2024
state_cd = 33
season_cd = 1
crop_plan_c = {
    91 : 40,
    3 : 40,
    5 : 40,
    6 : 40,
    15 : 40,
    33 : 40,
    11 : 40,
    12 : 40,
    92 : 60,
    14 : 40
}

crop_plan_s = {
    91 : 40,
    3 : 40,
    5 : 40,
    6 : 40,
    15 : 40,
    33 : 40,
    11 : 40,
    12 : 40,
    92 : 60,
    14 : 40
}


d_ST = {1 : "Central", 2 : "State"}

d_state_nm = {
	1 : "Jammu & Kashmir",
	2 : "Himachal Pradesh",
	3 : "Punjab",
	5 : "Uttarakhand",
	6 : "Haryana",
	8 : "Rajasthan",
	9 : "Uttar Pradesh",
	10 : "Bihar",
	18 : "Assam",
	19 : "West Bengal",
	20 : "Jharkhand",
	21 : "Odisha",
	22 : "Chattisgarh",
	23 : "Madhya Pradesh",
	24 : "Gujarat",
	27 : "Maharashtra",
	28 : "Andhra Pradesh",
	29 : "Karnataka",
	32 : "Kerala",
	33 : "Tamil Nadu",
	34 : "Puducherry",
	36 : "Telangana"
}



d_ro_nm = {
	11 : 'Jammu',
	12 : 'Srinagar',
	21 : 'Shimla',
	31 : 'Jalandhar',
	32 : 'Mohali',
	51 : 'Dehradun',
	61 : 'Chandigarh',
	81 : 'Ajmer',
	82 : 'Jaipur',
	91 : 'Agra',
	92 : 'Prayagraj',
	93 : 'Bareilly',
	94 : 'Lucknow',
	101 : 'Muzaffarpur',
	102 : 'Patna',
	181 : 'Guwahati',
	182 : 'Dibrugarh',
	191 : 'Burdwan',
	192 : 'Kolkata',
	193 : 'Maldah',
	201 : 'Ranchi',
	211 : 'Bhubaneswar',
	212 : 'Sambalpur',
	221 : 'Raipur',
	231 : 'Bhopal',
	232 : 'Gwalior',
	233 : 'Jabalpur',
	241 : 'Ahmedabad',
	242 : 'Vadodara',
	271 : 'C S Nagar',
	272 : 'Mumbai',
	273 : 'Nagpur',
	274 : 'Pune',
	281 : 'Kadapa',
	282 : 'Vijayawada',
	291 : 'Bengaluru',
	292 : 'Hubli',
	321 : 'Kozhikode',
	322 : 'Thiruvananthapuram',
	331 : 'Coimbatore',
	332 : 'Chennai',
	333 : 'Madurai',
	361 : 'Hyderabad'
}


d_state_season_nm = {
	1: {1: 'Kharif'},
	2: {1: 'Kharif', 4: 'Rabi'},
	3: {1: 'Kharif'},
	5: {1: 'Kharif', 4: 'Rabi'},
	6: {1: 'Kharif'},
	8: {1: 'Kharif'},
	9: {1: 'Kharif'},
	10: {2: 'Bhadai', 3: 'Aghani', 4: 'Rabi', 5: 'Summer'},
	18: {2: 'Early Kharif', 3: 'Late Kharif', 4: 'Rabi', 5: 'Summer'},
	19: {2: 'Early Kharif', 3: 'Late Kharif', 4: 'Rabi', 5: 'Summer'},
	20: {2: 'Bhadai', 3: 'Agahani'},
	21: {2: 'Autumn', 3: 'Winter', 5: 'Summer'},
	22: {1: 'Kharif', 4: 'Rabi'},
	23: {1: 'Kharif', 4: 'Rabi'},
	24: {1: 'Kharif', 4: 'Rabi', 5: 'Summer'},
	27: {1: 'Kharif'},
	28: {1: 'Kharif', 4: 'Rabi'},
	29: {1: 'Kharif'},
	32: {2: 'Autumn', 6: 'Whole Year'},
	33: {2: 'Phase-I', 3: 'Phase-II', 4: 'Phase-III'},
	34: {1: 'Kharif', 4: 'Rabi-I', 5: 'Rabi-II'},
	36: {1: 'Kharif', 4: 'Rabi'}
}



d_state_distt_nm = {
    1: {1: 'Kupwara',
	2: 'Baramula',
	4: 'Badgam',
	5: 'Pulwama',
	6: 'Anantnag',
	9: 'Doda',
	10: 'Udhampur',
	11: 'Poonch',
	12: 'Rajouri',
	13: 'Jammu',
	14: 'Kathua',
	15: 'Samba',
	16: 'Kishtwar',
	17: 'Reasi',
	18: 'Ramban',
	19: 'Ganderbal',
	20: 'Kulgam',
	22: 'Bandipora'},
2: {1: 'Chamba',
	2: 'Kangra',
	4: 'Kullu',
	5: 'Mandi',
	6: 'Hamirpur',
	7: 'Una',
	8: 'Bilaspur',
	9: 'Solan',
	10: 'Sirmaur',
	11: 'Shimla',
	12: 'Kinnour'},
3: {1: 'Gurdaspur',
	2: 'Amritsar',
	3: 'Kapurthala',
	4: 'Jalandhar',
	5: 'Hoshiarpur',
	6: 'Sbs Nagar',
	7: 'Ropar',
	8: 'Fatehgarh Sahib',
	9: 'Ludhiana',
	10: 'Moga',
	11: 'Ferozpur',
	12: 'Mukatsar',
	13: 'Faridkot',
	14: 'Bhatinda',
	15: 'Mansa',
	16: 'Sangrur',
	17: 'Patiala',
	18: 'Mohali',
	19: 'Barnala',
	20: 'Tarn Taran',
	21: 'Pathankot',
	22: 'Fazilka',
	23: 'Malerkotla'},
5: {1: 'Uttarkashi',
	2: 'Chamoli',
	3: 'Rudraprayag',
	4: 'Tehrigarhwal',
	5: 'Dehradun(Plains)',
	6: 'Pourigarhwal',
	7: 'Pithoragarh',
	8: 'Champawat',
	9: 'Almora',
	10: 'Bageshwar',
	11: 'Nainital(Plains)',
	12: 'Udhamsinghnagar',
	13: 'Haridwar',
	14: 'Nainital(Hills)',
	15: 'Dehradun(Hills)'},
6: {1: 'Panchkula',
	2: 'Ambala',
	3: 'Yamunanagar',
	4: 'Kurukshetra',
	5: 'Kaithal',
	6: 'Karnal',
	7: 'Panipat',
	8: 'Sonepat',
	9: 'Jind',
	10: 'Fatehabad',
	11: 'Sirsa',
	12: 'Hisar',
	13: 'Bhiwani',
	14: 'Rohtak',
	15: 'Jhajjar',
	16: 'Mahendergarh',
	17: 'Rewari',
	18: 'Gurugram',
	19: 'Faridabad',
	20: 'Mewat',
	21: 'Palwal',
	22: 'Charki Dadri'},
8: {1: 'Sri Ganganagar',
	2: 'Hanumangarh',
	3: 'Bikaner',
	4: 'Churu',
	5: 'Jhunjhunu',
	6: 'Alwar',
	7: 'Bharatpur',
	8: 'Dholpur',
	9: 'Karauli',
	10: 'Sawai Madhopur',
	11: 'Dausa',
	12: 'Jaipur ',
	13: 'Sikar',
	14: 'Nagaur',
	15: 'Jodhpur',
	16: 'Jaisalmer',
	17: 'Barmer',
	18: 'Jalore',
	19: 'Sirohi',
	20: 'Pali',
	21: 'Ajmer',
	22: 'Tonk',
	23: 'Bundi',
	24: 'Bhilwara',
	25: 'Rajsamand',
	26: 'Udaipur',
	27: 'Dungarpur',
	28: 'Banswara',
	29: 'Chittorgarh',
	30: 'Kota',
	31: 'Baran',
	32: 'Jhalawar',
	33: 'Pratapgarh',
	34: 'Balotra',
	35: 'Beawar',
	36: 'Deedvana And Kuchamn',
	37: 'Deeg',
	38: 'Phalodi',
	39: 'Khairthal-Tizara',
	40: 'Kotputli-Behror',
	41: 'Salumbar'},
9: {1: 'Saharanpur',
	2: 'Muzaffarnagar',
	3: 'Bijnor',
	4: 'Moradabad',
	5: 'Rampur',
	6: 'J. P. Nagar',
	7: 'Meerut',
	8: 'Bagpat',
	9: 'Ghaziabad',
	10: 'G.B. Nagar ',
	11: 'Bulandsehar',
	12: 'Aligarh',
	13: 'Mahamaya Nagar',
	14: 'Mathura',
	15: 'Agra',
	16: 'Firozabad',
	17: 'Etah',
	18: 'Mainpuri',
	19: 'Badaun',
	20: 'Bareilly',
	21: 'Pilibhit',
	22: 'Shahjahanpur',
	23: 'Khiri',
	24: 'Sitapur',
	25: 'Hardoi',
	26: 'Unnao',
	27: 'Lucknow',
	28: 'Raibareilly',
	29: 'Farrukhabad',
	30: 'Kannauj',
	31: 'Etawah',
	32: 'Auraiya',
	33: 'Kanpur Dehat',
	34: 'Kanpur Nagar',
	35: 'Jalaun',
	36: 'Jhansi',
	37: 'Lalitpur',
	38: 'Hamirpur',
	39: 'Mahoba',
	40: 'Banda',
	41: 'Chitrakoot',
	42: 'Fatehpur',
	43: 'Pratapgarh',
	44: 'Kaushambi',
	45: 'Prayagraj',
	46: 'Barabanki',
	47: 'Faizabad',
	48: 'Ambedkar Nagar',
	49: 'Sultanpur',
	50: 'Bahraich',
	51: 'Shrawasti',
	52: 'Balrampur',
	53: 'Gonda',
	54: 'Siddarthnagar',
	55: 'Basti',
	56: 'Sant Kabir Nagar',
	57: 'Maharajganj',
	58: 'Gorakhpur',
	59: 'Kushinagar',
	60: 'Deoria',
	61: 'Azamgarh',
	62: 'Mau',
	63: 'Baliya',
	64: 'Jaunpur',
	65: 'Ghazipur',
	66: 'Chandauli',
	67: 'Varanasi',
	68: 'Sant Ravidas Nagar',
	69: 'Mirzapur',
	70: 'Sonbhadra',
	71: 'Kasganj',
	72: 'Hapur',
	73: 'Amethi',
	74: 'Sambhal',
	75: 'Shamli'},
10: {1: 'W. Champaran',
	2: 'E.Champaran',
	3: 'Sheohar',
	4: 'Sitamarhi',
	5: 'Madhubani',
	6: 'Supaul',
	7: 'Araria',
	8: 'Kishanganj',
	9: 'Purnia',
	10: 'Katihar',
	11: 'Madhepura',
	12: 'Saharsa',
	13: 'Darbhanga',
	14: 'Muzaffarpur',
	15: 'Gopalganj',
	16: 'Siwan',
	17: 'Saran',
	18: 'Vaishali',
	19: 'Samastipur',
	20: 'Begusarai',
	21: 'Khagaria',
	22: 'Bhagalpur',
	23: 'Banka',
	24: 'Munger',
	25: 'Lakhisarai',
	26: 'Shekhpura',
	27: 'Nalanda',
	28: 'Patna',
	29: 'Bhojpur',
	30: 'Buxar',
	31: 'Bhabhua',
	32: 'Rohtash',
	33: 'Jehanabad',
	34: 'Aurangabad',
	35: 'Gaya',
	36: 'Nawada',
	37: 'Jamuai',
	38: 'Arwal'},
18: {1: 'Kokrajhar',
	2: 'Dhubri',
	3: 'Goalpara',
	4: 'Bongaigaon',
	5: 'Barpeta',
	6: 'Kamrup®',
	7: 'Nalbari',
	8: 'Darrang',
	9: 'Morigaon',
	10: 'Nagaon',
	11: 'Sonitpur',
	12: 'Lakhimpur',
	13: 'Dhemaji',
	14: 'Tinsukia',
	15: 'Dibrugarh',
	16: 'Sivasagar',
	17: 'Jorhat',
	18: 'Golaghat',
	19: 'Karbianglong',
	20: 'Dimo Hasao',
	21: 'Cachar',
	22: 'Karimganj',
	24: 'Chirang',
	25: 'Baksa',
	26: 'Kamrup(M)',
	27: 'Udalguri',
	28: 'Biswanath',
	30: 'Majuli',
	31: 'Hojai',
	32: 'West Karbianglong',
	33: 'South Salmara'},
19: {2: 'Jalpaiguri',
	3: 'Coochbihar',
	4: 'Uttar Dinajpur',
	5: 'Dakshin Dinajpur',
	6: 'Maldah',
	7: 'Murshidabad',
	8: 'Birbhum',
	9: 'Barddhaman (East)',
	10: 'Nadia',
	11: '24 Pargana( N)',
	12: 'Hoogly',
	13: 'Bankura',
	14: 'Purulia',
	15: 'Howrah',
	17: '24 Pargana (S)',
	18: 'Medinipur (W)',
	19: 'Medinipur (E)',
	20: 'Alipurduar',
	23: 'Jhargram'},
20: {1: 'Garhwa',
	2: 'Palamu',
	3: 'Chatra',
	4: 'Hazaribagh',
	6: 'Giridih',
	7: 'Deoghar',
	8: 'Godda',
	9: 'Sahebganj',
	10: 'Pakud',
	11: 'Dumka',
	12: 'Dhanbad',
	13: 'Bokaro',
	14: 'Ranchi',
	15: 'Lohardaga',
	16: 'Gumla',
	17: 'W.Singhbhum',
	18: 'E. Singhbhum',
	19: 'Latehar',
	20: 'Simdega',
	21: 'Jamtara',
	24: 'Khunti'},
21: {1: 'Bargarh',
	2: 'Jharsuguda',
	3: 'Sambalpur',
	4: 'Deogarh',
	5: 'Sundargarh',
	6: 'Keonjhar',
	7: 'Mayurbhanj',
	8: 'Baleshwar',
	9: 'Bhadrak',
	10: 'Kendrapada',
	11: 'Jagatsinghpur',
	12: 'Cuttack',
	13: 'Jajpur',
	14: 'Dhankenal',
	15: 'Angul',
	16: 'Nayagarh',
	17: 'Khurda',
	18: 'Puri',
	19: 'Ganjam',
	20: 'Gajapati',
	21: 'Kandhamal',
	22: 'Boudha',
	23: 'Subarnapur',
	24: 'Bolangir',
	25: 'Nuapada',
	26: 'Kalahandi',
	27: 'Rayagada',
	28: 'Nawrangpur',
	29: 'Koraput',
	30: 'Malkangiri'},
22: {1: 'Korea',
	2: 'Sarguja',
	3: 'Jashpur',
	4: 'Raigarh',
	5: 'Korba',
	7: 'Bilaspur',
	8: 'Kabirdham',
	9: 'Rajnandgaon',
	10: 'Durg',
	11: 'Raipur',
	12: 'Mahasamund',
	13: 'Dhamtri',
	14: 'Kanker',
	15: 'Baster',
	16: 'Dantewara',
	19: 'Sukma',
	20: 'Kondan Gaon',
	21: 'Baloda Bazar',
	22: 'Gariyaband',
	23: 'Bemetara',
	24: 'Balod',
	25: 'Mungeli',
	26: 'Surajpur',
	27: 'Balrampur',
	28: 'Mc Bharatpur',
	29: 'S Bilaigarh',
	31: 'Gorela Pendra',
	32: 'Kc Gandai',
	33: 'Mm Ambagarh Chowki'},
23: {1: 'Seopur',
	2: 'Murena',
	3: 'Bhind',
	4: 'Gwalior',
	5: 'Datia',
	6: 'Shivpuri',
	7: 'Guna',
	8: 'Tikamgarh',
	9: 'Chhatarpur',
	10: 'Panna',
	11: 'Sagar',
	12: 'Damoh',
	13: 'Satna',
	14: 'Rewa',
	15: 'Umaria',
	16: 'Shahdol',
	17: 'Sidhi',
	18: 'Neemuch',
	19: 'Mandsaur',
	20: 'Ratlam',
	21: 'Ujjain',
	22: 'Shajapur',
	23: 'Dewas',
	24: 'Jhabua',
	25: 'Dhar',
	26: 'Indore',
	27: 'Khargone',
	28: 'Barwani',
	29: 'Khandwa',
	30: 'Rajgarh',
	31: 'Vidisha',
	33: 'Sehore',
	34: 'Raisen',
	35: 'Betul',
	36: 'Harda',
	37: 'Hoshangabad',
	38: 'Katni',
	39: 'Jabalpur',
	40: 'Narsinghpur',
	41: 'Dindori',
	42: 'Mandla',
	43: 'Chhindwara',
	44: 'Seoni',
	45: 'Balaghat',
	46: 'Ashok Nagar',
	47: 'Anup Pur',
	48: 'Burhanpur',
	49: 'Alirajpur',
	50: 'Singrauli',
	51: 'Agar Malwa'},
24: {1: 'Kachchh',
	2: 'Banaskantha',
	3: 'Patan',
	4: 'Mehsana',
	5: 'Sabarkantha',
	6: 'Gandhinagar',
	7: 'Ahmedabad',
	8: 'Surendranagar',
	9: 'Rajkot',
	10: 'Jamnagar',
	11: 'Porbandar',
	12: 'Junagadh',
	13: 'Amreli',
	14: 'Bhavnagar',
	15: 'Anand',
	16: 'Kheda',
	17: 'Panchmahal',
	18: 'Dohad',
	19: 'Vadodara',
	20: 'Narmada',
	21: 'Bharuch',
	22: 'Surat',
	23: 'Dangs',
	24: 'Navsari',
	25: 'Valsad',
	26: 'Tapi',
	27: 'Arvalli',
	28: 'Botad',
	29: 'Chhota\nUdepur',
	30: 'Devbhumi Dwarka',
	31: 'Gir Somnath',
	32: 'Mahisagar',
	33: 'Morbi'},
27: {1: 'Nandurbar',
	2: 'Dhule',
	3: 'Jalgaon',
	4: 'Buldhana',
	5: 'Akola',
	6: 'Wasim',
	7: 'Amravati',
	14: 'Yeotmal',
	15: 'Nanded',
	16: 'Hingoli',
	17: 'Parbhani',
	18: 'Jalna',
	19: 'C S Nagar',
	20: 'Nasik',
	25: 'Pune',
	26: 'Ahilyanagar',
	27: 'Beed',
	28: 'Latur',
	29: 'Dharashiv',
	30: 'Solapur',
	31: 'Satara',
	34: 'Kolhapur',
	35: 'Sangali',
	36: 'Palghar'},
28: {1: 'Srikakulam',
	2: 'Vizianagaram',
	4: 'East Godavari',
	5: 'West Godavari',
	6: 'Krishna',
	7: 'Guntur',
	8: 'Prakasam',
	9: 'S.P.S.Nellore',
	10: 'Y.S.R.District',
	11: 'Kurnool',
	12: 'Anathapuram',
	13: 'Chittoor',
	14: 'Palnadu',
	15: 'Baptla',
	16: 'Tirupati',
	17: 'Annamayya',
	18: 'Nandyala',
	19: 'Sri Satya Sai',
	20: 'Alluri Seetha Ramaraju',
	21: 'Nandyala',
	22: 'Ntr',
	23: 'Palnadu',
	24: 'Kakinada',
	25: 'Parvathipuram Manyam',
	26: 'Ntr'},
29: {1: 'Belgavi',
	2: 'Bagalkot',
	3: 'Vijayapura',
	4: 'Kalaburgi',
	5: 'Bidar',
	6: 'Raichur',
	7: 'Koppal',
	8: 'Gadag',
	9: 'Dharwad',
	10: 'Uttara Kannada',
	11: 'Haveri',
	12: 'Bellary',
	13: 'Chitradurga',
	14: 'Davanagere',
	15: 'Shivamogga',
	16: 'Udupi',
	17: 'Chickmagaluru',
	18: 'Tumakuru',
	19: 'Kolar',
	21: 'Bengaluru(R)',
	22: 'Mandya',
	23: 'Hassan',
	24: 'Dakshina Kannada',
	25: 'Kodagu',
	26: 'Mysuru',
	27: 'Chamarajanagar',
	28: 'Ramnagara',
	29: 'Chikkaballapura',
	30: 'Yadgir',
	31: 'Vijayanagara'},
32: {1: 'Kasargod',
	2: 'Kannur',
	3: 'Wayanad',
	4: 'Kozhikode',
	5: 'Malapuram',
	6: 'Palakkad',
	7: 'Thrissur',
	8: 'Ernakulam',
	9: 'Idukki',
	10: 'Kottayam',
	11: 'Alappuzha',
	12: 'Pathanamthitta',
	13: 'Kollam',
	14: 'Thiruvananthapuram'},
33: {1: 'Thiruvallur',
	3: 'Kancheepuram',
	4: 'Vellore',
	5: 'Dharmapuri',
	6: 'Thiruvannamalai',
	7: 'Villupuram',
	8: 'Salem',
	9: 'Namakkal',
	10: 'Erode',
	12: 'Coimbatore',
	13: 'Dindigul',
	14: 'Karur',
	15: 'Tiruchirapalli',
	16: 'Perambalur',
	17: 'Ariyalur',
	18: 'Cuddalore',
	19: 'Nagapattinam',
	20: 'Thiruvarur',
	21: 'Thanjavur',
	22: 'Pudukottai',
	23: 'Sivagangai',
	24: 'Madurai',
	25: 'Theni',
	26: 'Virudhunagar',
	27: 'Ramanathapuram',
	28: 'Thoothukudi',
	29: 'Tirunelveli',
	30: 'Kanyakumari',
	31: 'Krishnagiri',
	32: 'Thiruppur',
	33: 'Chengalpattu',
	34: 'Kallakurichi',
	35: 'Ranipet',
	36: 'Tirupathur',
	37: 'Tenkasi',
	38: 'Myladuthurai'},
34: {2: 'Puducherry',
	4: 'Karaikal'},
36: {1: 'Adilabad',
	2: 'Komaram Bheem',
	3: 'Mancherial',
	4: 'Nirmal',
	5: 'Nizamabad',
	6: 'Jagtial',
	7: 'Peddapalli',
	8: 'Jayashankar',
	9: 'Bhadradri',
	10: 'Mahabubabad',
	11: 'Warangal Rural',
	12: 'Hanumakonda',
	13: 'Karimnagar',
	14: 'Rajanna',
	15: 'Kamareddy',
	16: 'Sangareddy',
	17: 'Medak',
	18: 'Siddipet',
	19: 'Jangaon',
	20: 'Yadadri',
	23: 'Rangareddy',
	24: 'Vikarabad',
	25: 'Mahbubnagar',
	26: 'Jogulamba',
	27: 'Wanaparthy',
	28: 'Nagarkurnool',
	29: 'Nalgonda',
	30: 'Suryapet',
	31: 'Khammam',
	33: 'Narayanpet'}}

d_state_ro = {
	1: [11, 12],
	2: [21],
	3: [31, 32],
	5: [51],
	6: [61],
	8: [81, 82],
	9: [91, 92, 93, 94],
	10: [101, 102],
	18: [181, 182],
	19: [191, 192, 193],
	20: [201],
	21: [211, 212],
	22: [221],
	23: [231, 232, 233],
	24: [241, 242],
	27: [271, 272, 273, 274],
	28: [281, 282],
	29: [291, 292],
	32: [321, 322],
	33: [331, 332, 333],
	34: [332],
	36: [361]
}



d_ro_distt = {
	11: [9, 10, 11, 12, 13, 14, 15, 16, 17, 18],
	12: [1, 2, 4, 5, 6, 19, 20, 22],
	21: [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12],
	31: [1, 2, 3, 4, 5, 6, 20, 21],
	32: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23],
	51: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
	61: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22],
	81: [14, 15, 16, 17, 18, 19, 20, 21, 24, 25, 26, 27, 28, 29, 33, 34, 35, 36, 38, 41],
	82: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 22, 23, 30, 31, 32, 37, 39, 40],
	91: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 29, 30, 31, 32, 71, 72],
	92: [43, 44, 45, 47, 48, 49, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 73],
	93: [1, 2, 3, 4, 5, 6, 19, 20, 21, 22, 23, 24, 25, 74, 75],
	94: [26, 27, 28, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 46, 50, 51, 52, 53],
	101: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
	102: [22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38],
	181: [1, 2, 3, 4, 5, 6, 7, 19, 20, 21, 22, 24, 25, 26, 32, 33],
	182: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 27, 28, 30, 31],
	191: [9, 10, 12, 13, 14, 18, 19, 23],
	192: [11, 15, 17],
	193: [2, 3, 4, 5, 6, 7, 8, 20],
	201: [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 24],
	211: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22],
	212: [1, 2, 3, 4, 5, 23, 24, 25, 26, 27, 28, 29, 30],
	221: [1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33],
	231: [25, 26, 27, 28, 29, 31, 33, 34, 35, 36, 37, 43, 45, 48],
	232: [1, 2, 3, 4, 5, 6, 7, 8, 9, 18, 19, 20, 21, 22, 23, 24, 30, 46, 49, 51],
	233: [10, 11, 12, 13, 14, 15, 16, 17, 38, 39, 40, 41, 42, 44, 47, 50],
	241: [1, 6, 7, 8, 9, 10, 11, 12, 13, 14, 28, 30, 31, 33],
	242: [2, 3, 4, 5, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 29, 32],
	271: [1, 2, 3, 15, 16, 17, 18, 19, 20, 27, 28],
	272: [36],
	273: [4, 5, 6, 7, 14],
	274: [25, 26, 29, 30, 31, 34, 35],
	281: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 23],
	282: [1, 2, 4, 5, 6, 20, 21, 22, 23, 24, 25, 26],
	291: [15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29],
	292: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 30, 31],
	321: [1, 2, 3, 4, 5, 6, 7],
	322: [8, 9, 10, 11, 12, 13, 14],
	331: [5, 8, 9, 10, 12, 14, 15, 16, 17, 22, 31, 32],
	332: [1, 2, 3, 4, 6, 7, 18, 33, 34, 35, 36],
	333: [13, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 37, 38],
	361: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33]
}

In [4]:
#set database file path
folder_path = Path('./software')
for file_path in folder_path.iterdir():
    if file_path.is_file():
        if 'NSSAS' in file_path.name:
            mdb_path = "./" + "/".join(str(file_path).split("\\"))

#create connection with mdb database
conn_str = (
    r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
    f'DBQ={mdb_path};'
)
conn = pyodbc.connect(conn_str)

#get data from the table - Sch20RC7
cols7 = ['RC', 'ST', 'YR', 'SESON', 'CROP', 'NEXP', 'NEXNR', 'CONFAC', 
         'STAT', 'DIST', 'VILL', 'RSV', 'EXPNO', 'STAGIN', 'REXML', 
         'CCE', 'CFC', 'RCSSN', 'E1', 'E2', 'E9', 'E10', 'E11', 'E12', 
         'ADH', 'QTYG', 'QTYP', 'E13', 'EQST', 'EQSB', 'EQSW', 'EQSP', 
         'E14', 'STYPE', 'E3', 'SVARC', 'SEEDR', 'E4', 'IRR', 'E5', 
         'IRRP', 'NIRR', 'FERTC', 'E6', 'QTN', 'CODN', 'QTP', 'CODP', 
         'QTK', 'CODK', 'MANUR', 'E7', 'PEST', 'E8', 'E15', 'REJ', 
         'CROPER', 'CROPCON', 'AFFECT']

sql_query7 = 'SELECT * FROM "Sch20RC7"'
rc7 = pd.read_sql(sql_query7, conn)[cols7]
rc7 = (rc7
    .drop_duplicates()
    .apply(pd.to_numeric, errors="coerce")
    .query("STAT == @state_cd and SESON == @season_cd")
    .assign(
        #adjusting cotton crop code
        YR = year+1
        ,CROP = lambda x : x['CROP'].replace({55: 12, 57: 56})
        ,REJ = lambda x : x['REJ'].apply(lambda y : 1 if y==1 else 0)
        ,RSV = lambda x: x['RSV'].where(x["RSV"].isin([0, 1, 2, 3, 9]), 0)
        ,REXML = lambda x: x["REXML"].where(
                    ~(x["STAGIN"].isin([2, 3]) & ~x["REXML"].isin([1, 2, 3, 4, 9])),
                    9
        )
        ,CFC = lambda x: x["CFC"].where(
                    ~(x["STAGIN"].isin([1, 2]) & ~x["CFC"].isin([1, 2, 3, 4])),
                    4
        )
        ,RCSSN = lambda x: x["RCSSN"].where(
            ~(x["STAGIN"].isin([1, 2]) & ~x["RCSSN"].isin([0, 1, 2, 3, 9])),
            0
        )
        ,EQST = lambda x: x["EQST"].where(
            ~(x["STAGIN"].isin([1, 2]) & x["EQST"].isna()),
            9
        )
        ,EQSB = lambda x: x["EQSB"].where(
            ~(x["STAGIN"].isin([1, 2]) & x["EQSB"].isna()),
            9
        )
        ,EQSW = lambda x: x["EQSW"].where(
            ~(x["STAGIN"].isin([1, 2]) & x["EQSW"].isna()),
            9
        )
        ,EQSP = lambda x: x["EQSP"].where(
            ~(x["STAGIN"].isin([1, 2]) & x["EQSP"].isna()),
            9
        )
        ,QTYG = lambda x: (
            x["QTYG"]
            .where(x["STAGIN"] != 2, None)
            .where(~(x["STAGIN"].eq(1) & x["CROPCON"].eq(4)), 0)
        )
        ,QTYP = lambda x: (
            x["QTYP"]
            .where(x["STAGIN"] != 2, None)
            .where(~(x["STAGIN"].eq(1) & x["CROPCON"].eq(4)), 0)
        )
        ,E13 = lambda x: x["E13"].where(x["STAGIN"] != 2, None)
        ,AFFECT = lambda x: x['AFFECT'].where(~((x['STAGIN'].isin([1,2])) & (x['CROPCON'].isin([1,2]))), 0)
        ,IRR = lambda x : np.where(x['NIRR']>0, 1, 0)
        ,CODN = lambda x : x['CODN'].where(~(x['QTN']>0), 1)
        ,CODP = lambda x : x['CODP'].where(~(x['QTP']>0), 1)
        ,CODK = lambda x : x['CODK'].where(~(x['QTK']>0), 1)
        ,FERTC = lambda x : np.where(((x['QTN']>0) | (x['QTP']>0) | (x['QTK']>0)), 1, 0)
        ,
    )
)

rc7.to_excel('rc.xlsx', sheet_name='RC7', index=False)
#with pd.ExcelWriter('RC1237_clean.xlsx', mode='a', engine='openpyxl') as w7:
#    rc7.to_excel(w7, sheet_name='RC7', index=False)

conn.close()

In [50]:
rc7.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 61 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   EXPTID   1620 non-null   int64  
 1   SRL      1620 non-null   int64  
 2   RC       1620 non-null   int64  
 3   ST       1620 non-null   int64  
 4   YR       1620 non-null   int64  
 5   SESON    1620 non-null   int64  
 6   CROP     1620 non-null   int64  
 7   NEXP     1601 non-null   float64
 8   NEXNR    0 non-null      float64
 9   CONFAC   0 non-null      float64
 10  STAT     1620 non-null   int64  
 11  DIST     1620 non-null   int64  
 12  VILL     1620 non-null   int64  
 13  RSV      1609 non-null   float64
 14  EXPNO    1620 non-null   int64  
 15  STAGIN   1620 non-null   int64  
 16  REXML    36 non-null     float64
 17  CCE      1600 non-null   float64
 18  CFC      1592 non-null   float64
 19  RCSSN    1592 non-null   float64
 20  E1       1601 non-null   float64
 21  E2       1601 

In [ ]:
def clean_rc7(year_int):
    yr_int = year_int - 2001
    year = str(year_int)
    
    mdb_path = r"C:/ICS_Scheme/AS2.0/AS"+year+"/NSSAS"+year+".mdb"
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={mdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - Sch20RC7
    sql_query = "SELECT * FROM Sch20RC7"
    rc7 = (
        pd
        .read_sql(sql_query, conn)
        .astype({'EXPTID' : 'int64'})
    )
    conn.close()
    return rc7

In [3]:
def granularize(rc1, rc2, rc3, rc4):
    pass

In [4]:
def generate_annex(grc1, grc2, grc3, grc4):
    pass

In [ ]:
if __name__ == "__main__":
    #read & clean RCs
    year_int = 2024            #2024 for A.Y. 2024-25
    season_cd = [2,3]              #1-Kharif, 4-Rabi
    rc1_novp = {
        1 : [0,0],                     ##season_1 planning [cen, state]
        2 : [260, 260],                ##season_2 planning [cen, state]
        3 : [260, 260],                ##season_3 planning [cen, state]
        4 : [260, 260],                ##season_4 planning [cen, state]
        5 : [0,0]                      ##season_5 planning [cen, state]
    }
    rc1_novcod = {
        1 : [0,0],                     ##season_1 NOVCOD [cen, state]
        2 : [260, 260],                ##season_2 NOVCOD [cen, state]
        3 : [260, 260],                ##season_3 NOVCOD [cen, state]
        4 : [260, 260],                ##season_4 NOVCOD [cen, state]
        5 : [0,0]                      ##season_5 NOVCOD [cen, state]
    }

    rc1, rc2, rc3 = clean_rc123(year_int, season_cd, rc1_novp, rc1_novcod)
    rc7 = clean_rc7(year_int, season_cd)

    #Granularize RCs
    grc1, grc2, grc3, grc7 = granularize(crc1, crc2, crc3, crc7)

    #Generate Annexure Data
    generate_annex(grc1, grc2, grc3, grc7)